In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
class SPLC(nn.Module):
    r""" SPLC loss as described in the paper "Simple Loss Design for Multi-Label Learning with Missing Labels "

    .. math::
        &L_{SPLC}^+ = loss^+(p)
        &L_{SPLC}^- = \mathbb{I}(p\leq \tau)loss^-(p) + (1-\mathbb{I}(p\leq \tau))loss^+(p)

    where :math:'\tau' is a threshold to identify missing label 
          :math:`$\mathbb{I}(\cdot)\in\{0,1\}$` is the indicator function, 
          :math: $loss^+(\cdot), loss^-(\cdot)$ refer to loss functions for positives and negatives, respectively.

    .. note::
        SPLC can be combinded with various multi-label loss functions. 
        SPLC performs best combined with Focal margin loss in our paper. Code of SPLC with Focal margin loss is released here.
        Since the first epoch can recall few missing labels with high precision, SPLC can be used ater the first epoch.
        Sigmoid will be done in loss. 

    Args:
        tau (float): threshold value. Default: 0.6
        change_epoch (int): which epoch to combine SPLC. Default: 1
        margin (float): Margin value. Default: 1
        gamma (float): Hard mining value. Default: 2
        reduction (string, optional): Specifies the reduction to apply to the output:
            ``'none'`` | ``'mean'`` | ``'sum'``. ``'none'``: no reduction will be applied,
            ``'mean'``: the sum of the output will be divided by the number of
            elements in the output, ``'sum'``: the output will be summed. Default: ``'sum'``

        """

    def __init__(self,
                 tau: float = 0.6,
                 change_epoch: int = 1,
                 margin: float = 1.0,
                 gamma: float = 2.0,
                 reduction: str = 'none') -> None:
        super(SPLC, self).__init__()
        self.tau = tau
        self.change_epoch = change_epoch
        self.margin = margin
        self.gamma = gamma
        self.reduction = reduction

        self.apply_SPLC = False

    def forward(self, logits: torch.Tensor, targets: torch.LongTensor ) -> torch.Tensor:
        """
        call function as forward

        Args:
            logits : The predicted logits before sigmoid with shape of :math:`(N, C)`
            targets : Multi-label binarized vector with shape of :math:`(N, C)`
            epoch : The epoch of current training.

        Returns:
            torch.Tensor: loss
        """
        # Subtract margin for positive logits
        logits = torch.where(targets == 1, logits-self.margin, logits)
        
        # SPLC missing label correction
        if self.apply_SPLC:
            targets = torch.where(
                torch.sigmoid(logits) > self.tau,
                torch.tensor(1), targets)
        
        pred = torch.sigmoid(logits)

        # Focal margin for postive loss
        pt = (1 - pred) * targets + pred * (1 - targets)
        focal_weight = pt**self.gamma

        los_pos = targets * F.logsigmoid(logits)
        los_neg = (1 - targets) * F.logsigmoid(-logits)

        loss = -(los_pos + los_neg)
        loss *= focal_weight

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

In [52]:
class FocalLoss(torch.nn.Module):
    """
    https://pytorch.org/vision/main/_modules/torchvision/ops/focal_loss.html

     Loss used in RetinaNet for dense detection: https://arxiv.org/abs/1708.02002.

    Args:
        inputs (Tensor): A float tensor of arbitrary shape.
                The predictions for each example.
        targets (Tensor): A float tensor with the same shape as inputs. Stores the binary
                classification label for each element in inputs
                (0 for the negative class and 1 for the positive class).
        alpha (float or Tensor): Weighting factor in range (0,1) to balance
                positive vs negative examples or -1 for ignore. Default: ``0.25``.
        gamma (float): Exponent of the modulating factor (1 - p_t) to
                balance easy vs hard examples. Default: ``2``.
        reduction (string): ``'none'`` | ``'mean'`` | ``'sum'``
                ``'none'``: No reduction will be applied to the output.
                ``'mean'``: The output will be averaged.
                ``'sum'``: The output will be summed. Default: ``'none'``.
    Returns:
        Loss tensor with the reduction option applied.
    """
    def __init__(self, alpha = 0.25, gamma= 2.0, reduction = "none", eps=1e-8, margin=None):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.eps = eps
        self.margin = margin

    def forward(self, inputs, targets):
        
        if self.margin is not None: # for Focal Margin Loss, we have to subtract the margin 
            # p2 = torch.clamp(torch.sigmoid(inputs - self.margin), min=self.eps, max=1-self.eps)
            # ce_loss = F.binary_cross_entropy(p2, targets, reduction="none")
            p = torch.clamp(torch.sigmoid(inputs - self.margin), min=self.eps, max=1-self.eps)
            ce_loss = F.binary_cross_entropy_with_logits(inputs - self.margin, targets, reduction="none")
        else:
            p = torch.clamp(torch.sigmoid(inputs), min=self.eps, max=1-self.eps)
            ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
        
        
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = ce_loss * ((1 - p_t) ** self.gamma)

        if self.alpha >= 0:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss

        # Check reduction option and return loss accordingly
        if self.reduction == "none":
            pass
        elif self.reduction == "mean":
            loss = loss.mean()
        elif self.reduction == "sum":
            loss = loss.sum()
        else:
            raise ValueError(
                f"Invalid Value for arg 'reduction': '{self.reduction} \n Supported reduction modes: 'none', 'mean', 'sum'"
            )
        
        return loss

In [72]:
outputs_dict = {"Xerostomia": torch.tensor([[-0.1], [.1], [1.5], [1.4], [0.9]]), "Dysphagia": torch.tensor([[0.4], [0.3], [0.2], [0.1], [0.8]])}
labels_dict = {"Xerostomia": torch.tensor([0, 1, 0, 1, 1]), "Dysphagia": torch.tensor([1, -1, 0, 1, 0])}

predictions = torch.stack(list(outputs_dict.values()), dim=1).type(torch.float32).squeeze() # transposed! so that num columns = num toxicities
targets = torch.stack(list(labels_dict.values()), dim=1).type(torch.float32)

loss_function = MLLSC(variant="Focal")
loss = loss_function(predictions, targets)

print(loss)

tensor([[0.1091, 0.0207],
        [0.0364, 0.7550],
        [0.8530, 0.1810],
        [0.0022, 0.0364],
        [0.0071, 0.4181]])


In [73]:
loss_function.apply_MLLSC = True
loss = loss_function(predictions, targets)

print(loss)

tensor([[0.1454, 0.0826],
        [0.1454, 0.1004],
        [0.0067, 0.1212],
        [0.0086, 0.1454],
        [0.0285, 0.0357]])


In [74]:
f_loss_func = FocalLoss(gamma=2, margin=0)
loss = f_loss_func(predictions, targets)
print(loss)

tensor([[0.1091, 0.0207],
        [0.0364, 0.7550],
        [0.8530, 0.1810],
        [0.0022, 0.0364],
        [0.0071, 0.4181]])


In [75]:
targets.shape

torch.Size([5, 2])

In [3]:
True * 1 + False * 2

1